# Documentation IA — Awalé Boissons

## 1. Objectif

Le composant IA du projet Awalé Boissons sert à analyser automatiquement les commentaires provenant de Facebook, Instagram et TikTok afin d'extraire une **Customer Voice** exploitable pour l'analyse marketing.

Les prédictions portent sur :

- `language_model` : langue du commentaire ;
- `sentiment_model` : sentiment ;
- `theme_model` : thème ;
- `product_model` : produit mentionné ;
- `is_spam_model` : détection du spam.

L'IA complète les analyses de dépenses, ventes et commandes. Elle ne remplace pas les données transactionnelles et ne permet pas, à elle seule, d'attribuer des ventes à un canal marketing.

---

## 2. Positionnement méthodologique

Le principe retenu est :

> **faits observés → signaux → limites → hypothèses → test**

Les classifications IA sont donc utilisées comme des **signaux de Customer Voice**, et non comme une preuve de causalité marketing.

Exemple :

- fait : de nombreux commentaires non-spam sont classés dans le thème `price` avec un sentiment négatif ;
- signal : le prix constitue un irritant récurrent dans la voix client ;
- limite : les commentaires ne démontrent pas qu'un canal marketing a causé cet irritant ;
- hypothèse : une campagne ou une offre peut être testée avec un mécanisme de suivi ;
- test : mesurer séparément exposition, conversion et ventes associées.

---

## 3. Données d'entrée

La source principale de l'inférence complète est :

`data/processed/social_comments_for_ai.csv`

Les commentaires sont également intégrés au pipeline dbt via :

`raw_social_comments_predictions_v2`

Puis enrichis dans :

`int_social_comments_enriched`

La jointure entre commentaires et prédictions est réalisée sur :

`comment_id`

Le grain attendu est :

> **1 ligne = 1 commentaire**

---

## 4. Modèle et stratégie d'inférence

### Modèle local

Le modèle utilisé pour le benchmark et l'inférence est :

`Qwen/Qwen2.5-0.5B-Instruct`

L'objectif est de produire des classifications structurées pour les cinq variables IA.

### Version hybride

Une version hybride a été développée afin de combiner :

- règles déterministes lorsque les conditions sont suffisamment explicites ;
- modèle local pour les cas non résolus.

Cependant, lors de l'inférence complète finale sur les 2 831 commentaires :

- commentaires traités : **2 831**
- règles seules : **0**
- modèle local utilisé : **2 831**
- `rules_complete = false` : **2 831**

Il faut donc présenter l'inférence finale comme une **inférence par modèle local**, et non comme une majorité de classifications par règles.

---

## 5. Benchmark humain

Un échantillon de **50 commentaires annotés manuellement** a servi de benchmark.

Important :

> Les 50 annotations humaines constituent un jeu d'évaluation et non un jeu de fine-tuning.

Le benchmark sert à mesurer l'accord entre les prédictions et les annotations humaines.

### Résultats

| Tâche | Qwen V1 | Version hybride |
|---|---:|---:|
| Language | 72 % | 22 % |
| Sentiment | 54 % | 80 % |
| Theme | 40 % | 78 % |
| Product | 58 % | 52 % |
| Spam | 94 % | 88 % |
| Exact agreement | 8 % | 14 % |

Ces résultats montrent que la version hybride améliore certaines dimensions, notamment le sentiment et le thème, mais dégrade d'autres dimensions. Ils ne justifient donc pas une affirmation générale selon laquelle une version serait meilleure pour toutes les tâches.

---

## 6. Pourquoi conserver un benchmark humain ?

Le benchmark permet de détecter :

- les erreurs de classification ;
- les catégories ambiguës ;
- les limites du modèle ;
- les régressions lors d'une future modification du prompt ou du modèle.

Il doit rester séparé des données d'inférence opérationnelle.

À chaque changement important du modèle, du prompt ou des règles, le benchmark doit être rejoué.

---

## 7. Résultats de l'inférence complète

Le jeu complet contient :

- **2 831 commentaires**
- Facebook : 941
- Instagram : 1 127
- TikTok : 763

Validation du fichier de prédictions :

- 2 831 lignes ;
- `comment_id` unique ;
- 0 doublon ;
- 0 valeur NULL dans les cinq champs de prédiction.

Validation DuckDB :

- 2 831 lignes ;
- 2 831 `comment_id` uniques ;
- 0 doublon ;
- 0 valeur NULL pour langue, sentiment, thème, produit et spam ;
- modèle utilisé sur 2 831 lignes.

---

## 8. Résultats Customer Voice

Sur l'ensemble des commentaires :

- 1 186 positifs ;
- 735 négatifs ;
- 513 spam.

Principaux thèmes :

| Thème | Nombre |
|---|---:|
| Taste | 680 |
| Price | 376 |
| Promotion | 313 |
| Availability | 170 |
| Packaging | 208 |
| Health | 152 |
| Delivery | 109 |
| Service | 1 |
| Other | 309 |

### Signaux principaux

En excluant le spam :

**Prix**
- 347 négatifs sur 376 commentaires du thème ;
- signal très majoritairement négatif.

**Disponibilité**
- 151 négatifs sur 170 ;
- signal très majoritairement négatif.

**Goût**
- 578 positifs ;
- 88 neutres ;
- 14 négatifs ;
- signal très majoritairement positif.

**Packaging**
- 145 positifs ;
- 61 négatifs ;
- 2 neutres.

**Promotion**
- 156 positifs ;
- 142 neutres ;
- 15 négatifs.

Ces résultats décrivent les commentaires classifiés. Ils ne constituent pas une mesure de satisfaction représentative de toute la clientèle.

---

## 9. Analyse par produit

Les produits détectés sont :

- `bissap`
- `gingembre`
- `bouye`
- `multiple`
- `unknown`
- `none`

Résultats sentiment × produit, hors spam :

| Produit | Positif | Négatif | Neutre |
|---|---:|---:|---:|
| Bissap | 211 | 41 | 10 |
| Bouye | 253 | 131 | 45 |
| Gingembre | 107 | 16 | 3 |
| Multiple | 2 | 38 | 4 |
| None | 105 | 39 | 52 |
| Unknown | 508 | 470 | 283 |

La catégorie `unknown` est importante : elle montre que l'identification automatique du produit reste une limite du système.

Il ne faut pas interpréter `unknown` comme un produit réel.

---

## 10. Intégration dans le pipeline

Le flux IA est :

```text
Social comments
      ↓
Préparation des commentaires
      ↓
Modèle IA local
      ↓
social_comments_predictions_v2_full.csv
      ↓
raw_social_comments_predictions_v2
      ↓
int_social_comments_enriched
      ↓
mart_social_monthly
      ↓
Dashboard / Customer Voice
```

Le modèle ne modifie pas les commentaires originaux.

Les prédictions sont conservées séparément puis jointes avec les données sociales via `comment_id`.

---

## 11. Contrôles qualité IA

Avant d'utiliser les prédictions, contrôler :

1. nombre de commentaires source ;
2. nombre de prédictions ;
3. unicité de `comment_id` ;
4. absence de doublons ;
5. absence de NULL dans les champs prédits ;
6. proportion de lignes effectivement traitées par le modèle ;
7. version du modèle ;
8. version du prompt/règles ;
9. durée d'exécution ;
10. coût si un modèle payant est utilisé.

Contrôle minimal :

```text
source_count == prediction_count
unique_comment_id == prediction_count
duplicate_comment_id == 0
NULL predictions == 0
```

---

## 12. Évaluation à maintenir

Le benchmark doit être suivi séparément de l'inférence complète.

Pour chaque nouvelle version :

- Accuracy par tâche ;
- Macro-F1 lorsque pertinent ;
- matrice de confusion ;
- erreurs principales ;
- exact agreement ;
- volume de données ;
- durée ;
- coût ;
- version du modèle ;
- version du prompt.

Une baisse sur une tâche ne doit pas être masquée par une amélioration sur une autre.

---

## 13. Limites

### Représentativité

Les commentaires sociaux ne représentent pas nécessairement toute la clientèle.

### Qualité des labels

Les prédictions sont des classifications automatiques et doivent être interprétées avec les performances du benchmark.

### Produit `unknown`

Une part importante des commentaires ne permet pas une identification automatique fiable du produit.

### Spam

Les commentaires détectés comme spam doivent être exclus des analyses de Customer Voice lorsque l'objectif est d'étudier la voix client.

### Causalité

L'IA ne permet pas de conclure :

> « tel canal a généré telle vente ».

Une telle conclusion nécessite un dispositif d'attribution ou d'expérimentation approprié.

---

## 14. Bonnes pratiques pour une nouvelle version

Avant de remplacer le modèle :

1. conserver l'ancien fichier de prédictions ;
2. conserver le benchmark humain ;
3. lancer le nouveau modèle sur le même benchmark ;
4. comparer les métriques par tâche ;
5. inspecter les erreurs ;
6. vérifier les NULL et doublons ;
7. lancer l'inférence complète ;
8. vérifier le volume final ;
9. exécuter `dbt test` ;
10. documenter la version retenue.

Ne jamais comparer deux versions sur des jeux de benchmark différents sans le signaler.

---

## 15. Reproductibilité

Depuis la racine du projet :

```bash
cd /mnt/c/Users/KSOMS/Favorites/awale_boissons
```

Préparer les données sociales, puis lancer l'inférence selon le script du projet.

Après chargement dans DuckDB :

```bash
cd dbt
dbt run
dbt test
```

Le pipeline final validé au moment de la livraison compte :

- **22 modèles dbt exécutés avec succès**
- **74 tests dbt PASS**
- **0 WARN**
- **0 ERROR**

---

## 16. Utilisation dans la décision marketing

La Customer Voice peut aider à :

- identifier les irritants récurrents ;
- détecter des signaux autour du prix et de la disponibilité ;
- suivre les réactions aux promotions ;
- observer les mentions de produits ;
- alimenter les questions à tester.

Elle ne doit pas être transformée en score automatique de rentabilité d'un canal.

La recommandation budgétaire de 15 M FCFA repose donc sur plusieurs sources :

- dépenses marketing ;
- media plan ;
- mesure disponible par canal ;
- ventes ;
- qualité des données ;
- Customer Voice ;
- conditions d'instrumentation.

L'IA est un **élément de preuve complémentaire**, pas le moteur unique de la décision.

---

## 17. Résumé exécutif

Le composant IA du projet fournit une classification automatisée de 2 831 commentaires sociaux selon cinq dimensions. Un benchmark humain indépendant de 50 commentaires permet d'évaluer les performances.

La version finale a été exécutée sur l'ensemble des commentaires et validée dans DuckDB. Les résultats Customer Voice mettent notamment en évidence des signaux négatifs associés aux thèmes `price` et `availability`, ainsi qu'un signal largement positif autour du `taste`.

Ces résultats doivent rester descriptifs. Pour transformer un signal en décision marketing, le projet applique systématiquement la chaîne :

> **signal → limite → hypothèse → test mesurable**

